# B2 · Descenso de gradiente a mano

**Nivel Intermedio** · Trabajo práctico B2 · Semana 4 · Se entrega al inicio de la clase de la semana 5 · Tiempo estimado: 3–4 horas · CPU (no hace falta GPU)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/tareas/B2_descenso_gradiente.ipynb)

Toda red neuronal se entrena con la misma idea: calcular hacia dónde sube la pérdida y dar un paso en la dirección contraria. En este trabajo programas ese algoritmo, el **descenso de gradiente** (*gradient descent*), sin ninguna librería que lo haga por ti. Lo pruebas en funciones de una y dos variables, dibujas su camino sobre un mapa de curvas de nivel, **lo rompes a propósito** con una mala tasa de aprendizaje y terminas ajustando una recta a datos médicos reales.

**Objetivos.** Al terminar podrás:

1. Programar el descenso de gradiente para funciones de una o varias variables y verificar un gradiente hecho a mano contra uno numérico.
2. Dibujar el camino del algoritmo sobre un gráfico de contornos y explicar lo que se ve (zigzag, avance lento, divergencia).
3. Encontrar, con experimentos y con una cuenta, la tasa de aprendizaje más grande con la que el algoritmo todavía converge.
4. Explicar por qué estandarizar las variables hace que el entrenamiento funcione.

**Prerrequisitos.** NumPy (semana 2: formas, `axis`, *broadcasting*) y gráficos con matplotlib (semana 3). La clase de la semana 4 (derivadas, gradiente, descenso de gradiente) ayuda mucho; si faltaste, la sección "Repaso rápido" de abajo trae lo necesario.

**Cómo entregar.** Antes de escribir nada: *Archivo → Guardar una copia en GitHub*, repositorio `Alto-Rendimiento-IA/entregas-<tu-usuario>`, rama `main`, ruta **`B2/B2_descenso_gradiente.ipynb`**, mensaje `Empezar B2`. Trabaja en la copia que se abre y guarda ahí después de cada ejercicio. Lo que esté en tu repositorio al inicio de la clase de la semana 5 es tu entrega.

**Uso de IA.** Puedes preguntarle a una IA qué es una curva de nivel o cómo se usa `plt.contour`. No le pidas el código del descenso de gradiente ni las respuestas de las preguntas: ese es el ejercicio.

## Preparación

Ejecuta esta celda primero. Carga las librerías, fija la semilla y carga los datos del ejercicio 6.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes

SEED = 0
rng = np.random.default_rng(SEED)

diabetes = load_diabetes(scaled=False)
bmi = diabetes.data[:, 2]  # índice de masa corporal (BMI) de 442 pacientes
target = diabetes.target   # una medida de cuánto avanzó la diabetes un año después
print("bmi:", bmi.shape, " target:", target.shape)

## Repaso rápido

- **Derivada.** $f'(x)$ es la pendiente de $f$ en $x$: cuánto cambia $f$ por cada poquito que se mueve $x$. Se aproxima con la **diferencia centrada**, $f'(x) \approx \dfrac{f(x+h) - f(x-h)}{2h}$ con $h \approx 10^{-5}$. Reglas: $(x^n)' = n x^{n-1}$, y la derivada de una suma es la suma de las derivadas.
- **Derivada parcial y gradiente.** Para $f(x, y)$, la derivada parcial $\partial f / \partial x$ deriva respecto de $x$ con $y$ quieta. El **gradiente** $\nabla f = (\partial f / \partial x,\, \partial f / \partial y)$ apunta hacia donde $f$ sube más rápido. Ejemplo: $f(x, y) = x^2 + 3y^2$ tiene $\nabla f = (2x,\, 6y)$.
- **Regla de la cadena.** Si $L = r^2$ y $r$ depende de $w$, entonces $\dfrac{\partial L}{\partial w} = 2r \cdot \dfrac{\partial r}{\partial w}$.
- **Descenso de gradiente.** Desde un punto inicial, se repite $\theta \leftarrow \theta - \eta \, \nabla f(\theta)$. El número $\eta$ es la **tasa de aprendizaje** (*learning rate*): el largo de cada paso.
- **Curvas de nivel.** Un gráfico de contornos (*contour plot*) dibuja las líneas donde $f$ vale lo mismo, como un mapa topográfico. Se hace así:

In [ ]:
gx, gy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-2, 2, 200))  # dos arreglos (200, 200)
fig, ax = plt.subplots(figsize=(6, 4))
cs = ax.contour(gx, gy, gx**2 + 3 * gy**2, levels=[0.5, 2, 4, 8, 12], cmap="viridis")
ax.clabel(cs, fontsize=8)                    # escribe el valor sobre cada curva
ax.plot([2, 1, 0.3], [1.5, 0.4, 0.05], "o-", color="tab:orange", label="un camino")
ax.set_aspect("equal")                       # misma escala en los dos ejes: los ángulos se ven bien
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

## Ejercicio 1: el gradiente numérico

Antes de confiar en un gradiente calculado a mano, conviene compararlo con uno numérico. Completa `numerical_grad(f, p, h=1e-5)`:

- `f` recibe un arreglo `p` de forma `(d,)` y devuelve un número.
- Devuelve un arreglo de forma `(d,)`: en la posición `i`, la diferencia centrada moviendo **solo** `p[i]`.

Puedes usar un bucle sobre las coordenadas. No modifiques `p`: trabaja sobre copias (`p.astype(float)` crea una).

In [ ]:
def numerical_grad(f, p, h=1e-5):
    # TODO
    raise NotImplementedError

In [ ]:
g = numerical_grad(lambda p: p[0] ** 2 + 3 * p[1] ** 2, np.array([1.0, 2.0]))
assert g.shape == (2,), f"forma {g.shape}, esperaba (2,)"
assert np.allclose(g, [2.0, 12.0], atol=1e-6), f"el gradiente de x² + 3y² en (1, 2) es (2, 12), dio {g}"
q = rng.normal(size=5)
q_before = q.copy()
assert np.allclose(numerical_grad(lambda v: np.sum(v**3), q), 3 * q**2, atol=1e-6), "el gradiente de Σ v³ es 3v²"
assert np.array_equal(q, q_before), "numerical_grad no debe modificar p"
print("✓ Ejercicio 1")

**Guarda en GitHub** con el mensaje `Ejercicio 1 resuelto`.

## Ejercicio 2: descenso de gradiente

Completa `gradient_descent(grad, x0, lr, steps)`:

- `grad` es una función que recibe un punto y devuelve el gradiente en ese punto.
- `x0` es el punto inicial: un número (una variable) o un arreglo de forma `(d,)` (varias variables).
- Da `steps` pasos de $x \leftarrow x - \eta \, \nabla f(x)$, con $\eta$ = `lr`.
- Devuelve **el camino completo**: un arreglo con el punto inicial y los `steps` puntos siguientes, con forma `(steps + 1,)` si `x0` es un número, o `(steps + 1, d)` si es un arreglo. Pista: guarda los puntos en una lista y al final conviértela con `np.array`.

La celda de verificación usa $f(x) = (x - 3)^2$, con $f'(x) = 2(x - 3)$ y mínimo en $x = 3$.

In [ ]:
def gradient_descent(grad, x0, lr, steps):
    # TODO
    raise NotImplementedError

In [ ]:
path = gradient_descent(lambda x: 2 * (x - 3), 0.0, 0.1, 50)
assert path.shape == (51,), f"forma {path.shape}, esperaba (51,): el punto inicial y 50 pasos"
assert np.isclose(path[0], 0.0) and np.isclose(path[1], 0.6), "desde 0 con η = 0.1, el primer paso va a 0.6"
assert abs(path[-1] - 3) < 1e-3, f"después de 50 pasos debería estar cerca de 3, está en {path[-1]:.4f}"
path2 = gradient_descent(lambda p: 2 * p, np.array([1.0, -2.0]), 0.25, 10)
assert path2.shape == (11, 2), f"con un punto de 2 coordenadas la forma es (11, 2), dio {path2.shape}"
assert np.allclose(path2[1], [0.5, -1.0]), "revisa el paso con varias variables"
print("✓ Ejercicio 2")

**Guarda en GitHub** con el mensaje `Ejercicio 2 resuelto`.

## Ejercicio 3: romperlo con una mala tasa de aprendizaje

Usamos $f(x) = 2x^2$, con mínimo en $x = 0$. Su derivada es $f'(x) = 4x$.

**3a.** Para cada tasa en `lrs`, corre 30 pasos desde $x_0 = 1$ y guarda el camino en el diccionario `paths` (la clave es la tasa). Después ejecuta la celda del gráfico: dibuja $|x_t|$, la distancia al mínimo, en escala logarítmica. En esa escala, una convergencia ordenada se ve como una recta que baja.

In [ ]:
def df3(x):
    return 4 * x


lrs = [0.01, 0.1, 0.25, 0.45, 0.5, 0.55]
paths = {}
# TODO 3a

In [ ]:
assert set(paths) == set(lrs) and all(len(p) == 31 for p in paths.values()), "un camino de 31 puntos por tasa"
fig, ax = plt.subplots(figsize=(8, 4.5))
for lr in lrs:
    ax.plot(np.abs(paths[lr]) + 1e-300, "o-", ms=3, label=f"η = {lr}")  # + 1e-300: log(0) no existe
ax.set_yscale("log")
ax.set_ylim(1e-17, 1e4)
ax.set_xlabel("paso t")
ax.set_ylabel("|x_t|  (distancia al mínimo)")
ax.set_title("Descenso de gradiente sobre f(x) = 2x² con distintas tasas")
ax.legend()
plt.show()

**3b.** Con experimentos, encuentra la tasa más grande con la que el algoritmo **todavía converge** (la distancia al mínimo después de 200 pasos es menor que $10^{-3}$). Prueba muchas tasas (por ejemplo con `np.linspace`, o achicando un intervalo a la mitad cada vez) y guarda la mayor que converge en `lr_max_experiment`. Busca con una precisión de 0.01 o mejor.

**3c.** Ahora con una cuenta. Escribe el paso del algoritmo para esta función: $x_{t+1} = x_t - \eta \cdot 4 x_t = (\;?\;)\, x_t$. Cada paso **multiplica** $x$ por el mismo número. ¿Para qué valores de $\eta$ ese número tiene valor absoluto menor que 1? Guarda el límite en `lr_max_theory` y explica tu cuenta en la celda de texto.

In [ ]:
lr_max_experiment = None  # TODO 3b
lr_max_theory = None      # TODO 3c

In [ ]:
assert lr_max_experiment is not None and lr_max_theory is not None, "completa 3b y 3c"
assert 0.48 <= lr_max_experiment <= 0.5, f"lr_max_experiment = {lr_max_experiment}: revisa tu búsqueda"
assert np.isclose(lr_max_theory, 0.5), "revisa la cuenta de 3c"
print("✓ Ejercicio 3")

*Tu cuenta de 3c, y qué pasa exactamente en el límite:*

-

**3d.** Si la función fuera $f(x) = \frac{c}{2} x^2$ (una parábola más cerrada cuanto más grande es $c$, su **curvatura**), ¿cuál sería la tasa máxima? Escribe la fórmula en función de $c$ y explica con tus palabras por qué una curva más cerrada pide pasos más cortos.

*Tu respuesta:*

-

**Guarda en GitHub** con el mensaje `Ejercicio 3 resuelto`.

## Ejercicio 4: varios valles

Hasta ahora cada función tenía un solo valle. La pérdida de una red neuronal tiene muchos. Usamos

$$f(x) = x^4 - 3x^2 + x$$

que tiene dos mínimos: uno **global** (el más bajo) cerca de $x = -1.30$ y uno **local** cerca de $x = 1.13$.

**4a.** Escribe `df4`, la derivada de $f$, a mano.

**4b.** Corre el descenso de gradiente desde cada punto de `starts`, con $\eta = 0.01$ y 300 pasos, y guarda el punto **final** de cada corrida en el arreglo `finals` (en el mismo orden que `starts`). Después ejecuta la celda del gráfico.

In [ ]:
def f4(x):
    return x**4 - 3 * x**2 + x


def df4(x):
    # TODO 4a
    raise NotImplementedError


starts = np.array([-2.0, -0.5, 0.1, 0.2, 2.0])
finals = None  # TODO 4b

In [ ]:
pts = np.array([-1.7, 0.4, 2.1])
assert np.allclose(df4(pts), [numerical_grad(lambda p: f4(p[0]), [v])[0] for v in pts], atol=1e-5), "revisa df4"
assert finals is not None and np.shape(finals) == starts.shape, "finals: un punto final por cada punto de partida"
assert np.isclose(finals[0], -1.3008, atol=1e-3) and np.isclose(finals[-1], 1.1309, atol=1e-3), "revisa las corridas"

xs = np.linspace(-2.1, 2.1, 300)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(xs, f4(xs), color="tab:blue", lw=2)
for x0, xf in zip(starts, finals):
    ax.annotate("", xy=(xf, f4(xf)), xytext=(x0, f4(x0)), arrowprops=dict(arrowstyle="->", color="tab:orange"))
    ax.plot(x0, f4(x0), "o", color="tab:orange")
    ax.plot(xf, f4(xf), "*", color="tab:red", ms=14)
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("Desde dónde empiezas decide a qué valle llegas")
plt.show()
for x0, xf in zip(starts, finals):
    print(f"desde {x0:5.1f} → termina en {xf:7.4f}, f = {f4(xf):7.4f}")
print("✓ Ejercicio 4")

**4c.** ¿Qué puntos de partida llegan al mínimo global y cuáles al local? Hay un punto que separa los dos grupos: ¿dónde está y qué tiene de especial la derivada ahí? (Mira la diferencia entre empezar en 0.1 y en 0.2.)

*Tu respuesta:*

-

**Guarda en GitHub** con el mensaje `Ejercicio 4 resuelto`.

## Ejercicio 5: dos variables y un valle alargado

Ahora una función de dos variables con un valle **alargado**: muy empinado en una dirección y casi plano en la otra.

$$f(x, y) = x^2 + 10\,y^2$$

**5a.** Escribe `grad5(p)`, el gradiente a mano, como `np.array([∂f/∂x, ∂f/∂y])`. La celda de verificación lo compara con tu `numerical_grad`.

In [ ]:
def f5(p):
    x, y = p
    return x**2 + 10 * y**2


def grad5(p):
    # TODO 5a
    raise NotImplementedError

In [ ]:
for _ in range(5):
    p = rng.normal(scale=3, size=2)
    assert np.allclose(grad5(p), numerical_grad(f5, p), atol=1e-5), f"grad5 no coincide con el gradiente numérico en {p}"
print("✓ Ejercicio 5a")

**5b.** Desde $p_0 = (-4, 1)$, corre 40 pasos con $\eta = 0.09$ y guárdalos en `path_big`, y 40 pasos con $\eta = 0.02$ en `path_small`.

**5c.** Haz **un** gráfico de contornos de $f_5$ en $[-5, 5] \times [-1.5, 1.5]$, con los dos caminos encima (cada uno de un color, con leyenda), el mínimo marcado, ejes con nombre y `ax.set_aspect("equal")`. Usa el ejemplo del "Repaso rápido" como modelo. Pista: para evaluar `f5` sobre la grilla, escribe la fórmula directamente con `gx` y `gy` (`gx**2 + 10 * gy**2`), porque `f5` espera un solo punto.

In [ ]:
p0 = np.array([-4.0, 1.0])
path_big = None    # TODO 5b
path_small = None  # TODO 5b

# TODO 5c: el gráfico de contornos con los dos caminos

In [ ]:
assert path_big is not None and path_big.shape == (41, 2), "path_big: 41 puntos de 2 coordenadas"
assert path_small is not None and path_small.shape == (41, 2), "path_small: 41 puntos de 2 coordenadas"
assert np.allclose(path_big[1], [-3.28, -0.8]), "revisa path_big"
assert np.allclose(path_small[1], [-3.84, 0.6]), "revisa path_small"
print(f"η = 0.09: termina en ({path_big[-1, 0]:.4f}, {path_big[-1, 1]:.4f}), f = {f5(path_big[-1]):.2e}")
print(f"η = 0.02: termina en ({path_small[-1, 0]:.4f}, {path_small[-1, 1]:.4f}), f = {f5(path_small[-1]):.2e}")
print("✓ Ejercicio 5b")

**5d.** Describe los dos caminos. ¿Cuál hace zigzag, y en qué dirección? ¿Cuál avanza lento, y en qué dirección? Relaciónalo con el ejercicio 3: en este valle, cada coordenada se comporta como una parábola $\frac{c}{2}x^2$ distinta. ¿Cuánto vale $c$ para $x$ y para $y$?

**5e.** Usando tu fórmula de 3d, ¿cuál es la tasa más grande con la que el descenso de gradiente converge en $f_5$? Compruébalo: corre 40 pasos con una tasa un poco mayor que ese límite y muestra qué pasa (un gráfico o un `print` alcanza). ¿Qué coordenada explota primero?

In [ ]:
# TODO 5e

*Tus respuestas a 5d y 5e:*

- 5d.
- 5e.

**Guarda en GitHub** con el mensaje `Ejercicio 5 resuelto`.

## Ejercicio 6: datos reales, y por qué estandarizar

El conjunto *diabetes* de sklearn tiene datos de 442 pacientes. Usamos una sola variable, el índice de masa corporal (`bmi`, entre 18 y 42), para predecir `target`, una medida de cuánto avanzó la enfermedad un año después. El modelo es una recta, $\hat{y} = w\,x + b$, y la pérdida es el error cuadrático medio (*mean squared error*, MSE):

$$L(w, b) = \frac{1}{n} \sum_{i=1}^{n} (w\,x_i + b - y_i)^2$$

**6a.** Completa `mse_and_grad(w, b, x, y)`: devuelve la tupla `(loss, dL_dw, dL_db)`, **sin bucles**. Pista con la regla de la cadena: con $r_i = w x_i + b - y_i$, la derivada de $r_i^2$ respecto de $w$ es $2 r_i x_i$, y respecto de $b$ es $2 r_i$. Después se promedia.

In [ ]:
def mse_and_grad(w, b, x, y):
    # TODO 6a
    raise NotImplementedError

In [ ]:
for w, b in [(0.0, 0.0), (5.0, -50.0), (12.0, 10.0)]:
    loss, dw, db = mse_and_grad(w, b, bmi, target)
    assert np.isclose(loss, np.mean((w * bmi + b - target) ** 2)), "revisa la pérdida"
    num = numerical_grad(lambda p: np.mean((p[0] * bmi + p[1] - target) ** 2), np.array([w, b]))
    assert np.allclose([dw, db], num, rtol=1e-5), f"en (w, b) = ({w}, {b}) el gradiente no coincide con el numérico"
print("✓ Ejercicio 6a")

**6b. Las referencias.** Antes de entrenar, dos números contra los cuales comparar:

- la **línea base**: predecir siempre la media de `target`, sin mirar el BMI (`baseline_mse`);
- el **mínimo exacto**: para una recta existe una fórmula cerrada, la de mínimos cuadrados, que calcula `np.polyfit(bmi, target, 1)` (devuelve `[w, b]`). Guarda su MSE en `best_mse`.

In [ ]:
baseline_mse = None  # TODO 6b
w_best, b_best = None, None  # TODO 6b
best_mse = None  # TODO 6b

In [ ]:
assert np.isclose(baseline_mse, target.var()), "la línea base predice siempre la media: su MSE es la varianza"
assert np.isclose(best_mse, 3890.457, atol=0.01), "revisa best_mse"
print(f"línea base:   MSE = {baseline_mse:.1f}")
print(f"np.polyfit:   MSE = {best_mse:.1f}  (w = {w_best:.3f}, b = {b_best:.3f})")
print("✓ Ejercicio 6b")

**6c. Con los datos crudos.** Entrena con `bmi` tal como viene, desde $w = 0$, $b = 0$, durante 2000 pasos. Primero busca la tasa de aprendizaje más grande que **no diverge** (prueba valores como 0.01, 0.005, 0.002, 0.001…) y guárdala en `lr_raw`. Con esa tasa, guarda la pérdida de cada paso en la lista `losses_raw` y los parámetros finales en `w_raw`, `b_raw`.

Vas a notar algo raro: no diverge, pero tampoco llega al mínimo de `np.polyfit`.

In [ ]:
lr_raw = None  # TODO 6c
losses_raw = []
w_raw, b_raw = 0.0, 0.0
# TODO 6c: 2000 pasos de descenso de gradiente sobre (w_raw, b_raw)

In [ ]:
assert lr_raw is not None and len(losses_raw) == 2000, "completa 6c: 2000 pérdidas"
assert np.isfinite(losses_raw[-1]) and losses_raw[-1] < losses_raw[0], "con esa tasa la pérdida no bajó: prueba una más chica"
print(f"datos crudos, η = {lr_raw}: MSE final = {losses_raw[-1]:.1f}  (w = {w_raw:.3f}, b = {b_raw:.3f})")
print(f"np.polyfit:              MSE      = {best_mse:.1f}  (w = {w_best:.3f}, b = {b_best:.3f})")
print("✓ Ejercicio 6c")

**6d. Con los datos estandarizados.** Estandariza el BMI, `bmi_std = (bmi - media) / desviación estándar`, y repite: desde $(0, 0)$, 100 pasos con $\eta = 0.1$. Guarda las pérdidas en `losses_std` y los parámetros finales en `w_std`, `b_std`.

La recta aprendida usa la variable estandarizada. Para compararla con `np.polyfit`, pásala a las unidades originales: si $x_{std} = (x - \mu)/\sigma$, entonces $w\,x_{std} + b = \dfrac{w}{\sigma}\,x + \left(b - \dfrac{w \mu}{\sigma}\right)$. Guarda esos dos números en `w_back`, `b_back`.

In [ ]:
bmi_std = None  # TODO 6d
losses_std = []
w_std, b_std = 0.0, 0.0
# TODO 6d: 100 pasos con η = 0.1

w_back, b_back = None, None  # TODO 6d

In [ ]:
assert bmi_std is not None and np.isclose(bmi_std.mean(), 0) and np.isclose(bmi_std.std(), 1), "bmi_std debe tener media 0 y desviación 1"
assert len(losses_std) == 100, "completa 6d: 100 pérdidas"
assert np.isclose(w_back, w_best, rtol=1e-3) and np.isclose(b_back, b_best, rtol=1e-3), "pasada a unidades originales, la recta debe coincidir con np.polyfit"
print(f"estandarizado, 100 pasos: MSE = {losses_std[-1]:.1f} → en unidades originales w = {w_back:.3f}, b = {b_back:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(losses_raw, label=f"datos crudos, η = {lr_raw}")
ax1.plot(losses_std, label="estandarizados, η = 0.1")
ax1.axhline(baseline_mse, color="gray", ls="--", label="línea base (la media)")
ax1.axhline(best_mse, color="black", ls=":", label="mínimo (np.polyfit)")
ax1.set_xscale("log")
ax1.set_ylim(3500, 12000)
ax1.set_xlabel("paso (escala logarítmica)")
ax1.set_ylabel("MSE")
ax1.set_title("Curvas de aprendizaje")
ax1.legend()
ax2.scatter(bmi, target, s=10, alpha=0.5, color="gray")
xs = np.array([bmi.min(), bmi.max()])
ax2.plot(xs, w_raw * xs + b_raw, lw=3, label="crudos, 2000 pasos")
ax2.plot(xs, w_back * xs + b_back, lw=3, ls="--", label="estandarizados, 100 pasos")
ax2.set_xlabel("índice de masa corporal (BMI)")
ax2.set_ylabel("avance de la enfermedad (target)")
ax2.set_title("Las dos rectas aprendidas")
ax2.legend()
plt.tight_layout()
plt.show()
print("✓ Ejercicio 6d")

**6e. ¿Por qué?** Haz dos gráficos de contornos de la pérdida $L(w, b)$, uno al lado del otro:

1. Datos crudos: $w \in [-5, 25]$, $b \in [-300, 300]$, con el camino de tu entrenamiento de 6c encima (guarda los puntos $(w, b)$ de cada paso; puedes volver a correrlo aquí).
2. Datos estandarizados: $w \in [-20, 110]$, $b \in [-20, 250]$, con el camino de 6d.

Pista para la grilla: si `Wg` y `Bg` tienen forma `(m, m)`, la pérdida en cada punto de la grilla es `np.mean((Wg[..., None] * x + Bg[..., None] - y) ** 2, axis=-1)`: el *broadcasting* agrega el eje de los pacientes al final y el promedio lo elimina. Usa niveles como `levels=np.geomspace(4000, 200000, 12)` (crudos) y `levels=np.geomspace(4000, 40000, 10)` (estandarizados): niveles espaciados de forma geométrica dejan ver el fondo del valle.

In [ ]:
# TODO 6e

**6f.** Explica con tus gráficos por qué con los datos crudos el entrenamiento se estanca: ¿qué forma tiene el valle, en qué dirección quedó atascado el camino, y por qué no se puede arreglar simplemente subiendo la tasa de aprendizaje? (Conecta con el ejercicio 5.) ¿Qué cambia al estandarizar?

*Tu respuesta:*

-

**Guarda en GitHub** con el mensaje `Ejercicio 6 resuelto`.

## Ejercicio 7: interpretar

Responde con tus números y tus gráficos:

1. En el ejercicio 6, ¿cuánto mejora la recta a la línea base? Exprésalo como porcentaje del MSE de la línea base. ¿Te parece que el BMI solo alcanza para predecir bien el avance de la enfermedad? ¿Qué otra información usarías?
2. Resume en tres frases lo que aprendiste sobre la tasa de aprendizaje: qué pasa si es muy chica, si es muy grande, y de qué depende el límite.
3. En una red neuronal, la pérdida depende de millones de parámetros y no podemos dibujarla. Con lo que viste en los ejercicios 4 y 6, nombra dos cosas que pueden salir mal al entrenar y qué harías para notarlas mirando la curva de aprendizaje.

*Tus respuestas:*

1.
2.
3.

**Guarda en GitHub** con el mensaje `Ejercicio 7 resuelto`.

## Conclusión

*Escribe aquí un párrafo en lenguaje sencillo: qué hace el descenso de gradiente, qué significan tus números del ejercicio 6 comparados con la línea base, y qué te gustaría probar después. Sin este párrafo el trabajo no tiene nota.*

## Uso de IA

*Una línea: para qué usaste IA en este trabajo, o "No usé IA".*

## Rúbrica

Primero asígnate tu puntaje (columna "Yo"). Tu compañero revisor completa la columna "Revisor" con un comentario en la copia de Drive que le compartas.

| Criterio | Qué se mira | Yo (0–2) | Revisor (0–2) |
|---|---|---|---|
| **Se ejecuta** | *Entorno de ejecución → Reiniciar y ejecutar todo* corre sin errores | | |
| **Es correcto** | Todas las verificaciones imprimen ✓; el gráfico de 5c y los dos de 6e muestran los caminos sobre los contornos, con ejes y leyenda | | |
| **Está justificado** | Las respuestas de 3c, 3d, 4c, 5d, 5e, 6f y del ejercicio 7 explican el porqué con tus números y gráficos | | |
| **Está bien comunicado** | La conclusión se entiende sin leer el código; hay un *commit* por ejercicio con un mensaje claro | | |
| **Total** | | /8 | /8 |

**Antes de la clase de la semana 5**: ejecuta *Entorno de ejecución → Reiniciar y ejecutar todo*, revisa que todo imprima ✓ y guarda por última vez en `B2/B2_descenso_gradiente.ipynb`.